# Dores

## Gerente Financeiro: 
"O gasto com atendimento passou de R$ 1,4 milhão nos últimos dezenove meses e cresce todo mês. O que mais me incomoda é que o crescimento não acompanha a receita. Precisamos cortar essa despesa."

In [0]:
from pyspark.sql import functions as F

SILVER = "v_credit_medallion.silver"

chamados = spark.table(f"{SILVER}.chamados")
custos = spark.table(f"{SILVER}.custos").select("id_chamado", "custo")
motivos = spark.table(f"{SILVER}.base_motivos").select("id_motivo", "nome_motivo", "categoria")

# Coeficiente de satisfação por motivo no começo de 2026
chamados_hora = spark.table(f"{SILVER}.chamados_hora").select("id_chamado", "hora_abertura_chamado")
nota_por_chamado = (
    spark.table(f"{SILVER}.pesquisa_satisfacao")
    .groupBy("id_chamado")
    .agg(F.round(F.avg("nota_atendimento"), 2).alias("nota_chamado"))
)

satisfacao_2026 = (
    chamados
    .join(chamados_hora, on="id_chamado", how="left")
    .join(nota_por_chamado, on="id_chamado", how="left")
    .join(motivos, on="id_motivo", how="left")
    .filter(
        (F.col("hora_abertura_chamado") >= F.lit("2026-01-01"))
        & (F.col("hora_abertura_chamado") < F.lit("2026-04-01"))
    )
    .groupBy("nome_motivo")
    .agg(
        F.round(
            F.avg(
                F.when(F.col("nota_chamado") >= 4, 1.0)
                 .when(F.col("nota_chamado").isNotNull(), 0.0)
            ) * 100,
            1,
        ).alias("satisfacao_2026_pct"),
    )
)

base_gastos = (
    chamados.join(custos, on="id_chamado", how="left")
    .join(motivos, on="id_motivo", how="left")
    .filter(F.col("custo").isNotNull())
)

# Gastos por motivo específico (top 15)
gastos_motivo = (
    base_gastos
    .groupBy("categoria", "nome_motivo")
    .agg(
        F.round(F.sum("custo"), 2).alias("custo_total"),
        F.round(F.avg("custo"), 2).alias("custo_medio"),
        F.count("*").alias("chamados"),
    )
    .join(satisfacao_2026, on="nome_motivo", how="left")
    .orderBy(F.desc("custo_total"))
    .limit(15)
)

gastos_motivo.display()

Os principais gastos se concentram em:

- Problema com aplicativo
- Compra não autorizada
- Renegociação de dívida
- Contestação de fatura

Esses motivos representam o pior cenário possível, sendo caros por chamada e tendo alto volume, somando aproximadamente R$ 779 mil. 
Também são justamente os motivos com os piores coeficientes de satisfação (50,8% a 56,6% em 2026). Ou seja, em um plano de reestruturação, estes casos deveriam ter prioridade.

## Diretora de Marketing: 
"No começo de 2025 nossos clientes nos amavam, o coeficiente de satisfação da V-Credit era 85%. Sentimos uma piora recente, mas acredito que esteja ligada a fatores externos e não à nossa operação. Não deve estar muito diferente do que vimos no ano passado."

In [0]:
chamados_hora = spark.table(f'{SILVER}.chamados_hora')
display(chamados_hora)

In [0]:
from pyspark.sql import functions as F

SILVER = "v_credit_medallion.silver"

chamados = spark.table(f"{SILVER}.chamados")
chamados_hora = spark.table(f"{SILVER}.chamados_hora").select("id_chamado", "hora_abertura_chamado","tempo_espera_informado","tempo_atendimento_informado")
custos = spark.table(f"{SILVER}.custos").select("id_chamado", "custo")
motivos = spark.table(f"{SILVER}.base_motivos").select("id_motivo", "nome_motivo", "categoria")

nota_por_chamado = (
    spark.table(f"{SILVER}.pesquisa_satisfacao")
    .groupBy("id_chamado")
    .agg(F.round(F.avg("nota_atendimento"), 2).alias("nota_chamado"))
)
base = (
     chamados 
    .join(chamados_hora, on="id_chamado", how="left")
    .join(custos, on="id_chamado", how="left")
    .join(nota_por_chamado, on="id_chamado", how="left")
    .join(motivos, on="id_motivo", how="left")
    .withColumn(
        "periodo",
        F.when(
            (F.col("hora_abertura_chamado") >= F.lit("2025-01-01"))
            & (F.col("hora_abertura_chamado") < F.lit("2025-04-01")),
            "Começo de 2025",
        ).when(
            (F.col("hora_abertura_chamado") >= F.lit("2026-01-01"))
            & (F.col("hora_abertura_chamado") < F.lit("2026-04-01")),
            "Começo de 2026",
        ),
    )
    .filter(F.col("periodo").isNotNull())
)

# Painel geral: principais indicadores por período
comparativo_periodo = (
    base
    .groupBy("periodo")
    .agg(
        F.count("*").alias("chamados"),
        F.count("nota_chamado").alias("chamados_avaliados"),
        F.round(F.avg("nota_chamado"), 2).alias("nota_media"),
        F.round(
            F.avg(
                F.when(F.col("nota_chamado") >= 4, 1.0)
                 .when(F.col("nota_chamado").isNotNull(), 0.0)
            ) * 100,
            1,
        ).alias("satisfacao_pct"),
    )
    .orderBy("periodo")
)

comparativo_periodo.display()

# Diferenças por indicador: começo de 2025 vs começo de 2026
resumo = (
    base
    .groupBy()
    .pivot("periodo", ["Começo de 2025", "Começo de 2026"])
    .agg(
        F.count("id_chamado").cast("double").alias("chamados"),
        F.round(F.avg(F.when(F.col("resolvido") == 1, 1.0).otherwise(0.0)) * 100, 1).alias("resolucao_pct"),
        F.round(F.avg("tempo_espera_informado"), 1).alias("espera_media_s"),
        F.round(F.avg("tempo_atendimento_informado"), 1).alias("atendimento_media_s"),
        F.round(F.avg("custo"), 2).cast("double").alias("custo_medio"),
        F.round(F.sum("custo"), 2).cast("double").alias("custo_total"),
        F.round(F.avg("nota_chamado"), 2).alias("nota_media"),
        F.round(
            F.avg(
                F.when(F.col("nota_chamado") >= 4, 1.0)
                 .when(F.col("nota_chamado").isNotNull(), 0.0)
            ) * 100,
            1,
        ).alias("coeficiente_pct"),
    )
)

diferencas = (
    resumo
    .select(
        F.expr("""
            stack(8,
              'Volume de chamados', `Começo de 2025_chamados`, `Começo de 2026_chamados`,
              'Taxa de resolução (%)', `Começo de 2025_resolucao_pct`, `Começo de 2026_resolucao_pct`,
              'Espera média (s)', `Começo de 2025_espera_media_s`, `Começo de 2026_espera_media_s`,
              'Atendimento médio (s)', `Começo de 2025_atendimento_media_s`, `Começo de 2026_atendimento_media_s`,
              'Custo médio (R$)', `Começo de 2025_custo_medio`, `Começo de 2026_custo_medio`,
              'Custo total (R$)', `Começo de 2025_custo_total`, `Começo de 2026_custo_total`,
              'Nota média', `Começo de 2025_nota_media`, `Começo de 2026_nota_media`,
              'Coeficiente satisfação (%)', `Começo de 2025_coeficiente_pct`, `Começo de 2026_coeficiente_pct`
            ) as (indicador, inicio_2025, inicio_2026)
        """)
    )
    .withColumn("diferenca", F.round(F.col("inicio_2026") - F.col("inicio_2025"), 2))
    .withColumn(
        "variacao_pct",
        F.round((F.col("inicio_2026") - F.col("inicio_2025")) / F.col("inicio_2025") * 100, 1),
    )
    .orderBy(F.desc("variacao_pct"))
)

# Linha extra: número de atendentes ativos em cada período
atendentes = spark.table(f"{SILVER}.base_atendentes")

headcount = (
    atendentes
    .groupBy()
    .agg(
        F.count_if(
            (F.col("data_admissao") <= F.lit("2025-04-01"))
            & (F.col("data_desligamento").isNull() | (F.col("data_desligamento") >= F.lit("2025-01-01")))
        ).cast("double").alias("inicio_2025"),
        F.count_if(
            (F.col("data_admissao") <= F.lit("2026-04-01"))
            & (F.col("data_desligamento").isNull() | (F.col("data_desligamento") >= F.lit("2026-01-01")))
        ).cast("double").alias("inicio_2026"),
    )
    .select(F.lit("Atendentes ativos").alias("indicador"), "inicio_2025", "inicio_2026")
    .withColumn("diferenca", F.round(F.col("inicio_2026") - F.col("inicio_2025"), 2))
    .withColumn(
        "variacao_pct",
        F.round((F.col("inicio_2026") - F.col("inicio_2025")) / F.col("inicio_2025") * 100, 1),
    )
)

# Estimativa: atendentes necessários em 2026 para manter a carga de 2025
ativos_2025 = atendentes.agg(
    F.count_if(
        (F.col("data_admissao") <= F.lit("2025-04-01"))
        & (F.col("data_desligamento").isNull() | (F.col("data_desligamento") >= F.lit("2025-01-01")))
    ).cast("double").alias("inicio_2025"),
)

estimativa = (
    resumo
    .join(ativos_2025, how="cross")
    .select(
        F.lit("Atendentes necessários (carga 2025)").alias("indicador"),
        F.col("inicio_2025"),
        F.round(
            F.col("Começo de 2026_chamados")
            / (F.col("Começo de 2025_chamados") / F.col("inicio_2025")),
            1,
        ).alias("inicio_2026"),
    )
    .withColumn("diferenca", F.round(F.col("inicio_2026") - F.col("inicio_2025"), 2))
    .withColumn(
        "variacao_pct",
        F.round((F.col("inicio_2026") - F.col("inicio_2025")) / F.col("inicio_2025") * 100, 1),
    )
)

# Chamadas médias por atendente em cada período
ativos = atendentes.agg(
    F.count_if(
        (F.col("data_admissao") <= F.lit("2025-04-01"))
        & (F.col("data_desligamento").isNull() | (F.col("data_desligamento") >= F.lit("2025-01-01")))
    ).cast("double").alias("ativos_2025"),
    F.count_if(
        (F.col("data_admissao") <= F.lit("2026-04-01"))
        & (F.col("data_desligamento").isNull() | (F.col("data_desligamento") >= F.lit("2026-01-01")))
    ).cast("double").alias("ativos_2026"),
)

carga_por_atendente = (
    resumo
    .join(ativos, how="cross")
    .select(
        F.lit("Chamados por atendente").alias("indicador"),
        F.round(F.col("Começo de 2025_chamados") / F.col("ativos_2025"), 1).alias("inicio_2025"),
        F.round(F.col("Começo de 2026_chamados") / F.col("ativos_2026"), 1).alias("inicio_2026"),
    )
    .withColumn("diferenca", F.round(F.col("inicio_2026") - F.col("inicio_2025"), 2))
    .withColumn(
        "variacao_pct",
        F.round((F.col("inicio_2026") - F.col("inicio_2025")) / F.col("inicio_2025") * 100, 1),
    )
)

diferencas.unionByName(headcount).unionByName(carga_por_atendente).unionByName(estimativa).orderBy(F.desc("variacao_pct")).display()

Entre 2025 e 2026 houve um aumento de 73% do volume de chamados, com isso seria necessário pelo menos 35 atendentes para que a qualidade se mantivesse a mesma de 2025. Como só havia 21 atendentes em 2026, a carga por atendente subiu em 64,9%, resultando em um também aumento de 73% no tempo de espera, queda de 7,9% nas resoluções e queda de 22,8% na satisfação.